# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library. This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples, including fields such as accession, description, coverage, peptide counts, molecular weight, pI, and normalized abundances.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Croissant schema URL**:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

> All entities (record sets, fields, columns, etc.) are referenced by their `@id` fields, as per FAIR and Croissant best practices.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and dataset structure from the Croissant schema URL using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show top-level metadata
print(f"Name: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"Description: {dataset.metadata.description}")
print(f"Keywords: {dataset.metadata.keywords}")


## 2. Data Overview

Review available record sets and the fields within each one, referencing their `@id` fields. This helps to identify which structures are present and what information can be extracted.

In [ ]:
# List all record sets and their available fields
print("Record Sets (by @id):\n")
for record_set in dataset.record_sets:
    print(f"- {record_set['@id']} : {record_set.get('name', '(no name)')}")
    print("  Fields:")
    for field in record_set['fields']:
        field_name = field.get('name', '(no name)')
        print(f"    - {field['@id']}: {field_name}")
    print()
# For this dataset, let's list those out as a table as well
record_sets_overview = []
for rs in dataset.record_sets:
    for field in rs['fields']:
        record_sets_overview.append({
            'record_set_id': rs['@id'],
            'record_set_name': rs.get('name', ''),
            'field_id': field['@id'],
            'field_name': field.get('name', '')
        })
pd.DataFrame(record_sets_overview)

## 3. Data Extraction

Load records from one or more record sets into a pandas DataFrame for further processing. Use the `@id` fields discovered above. For this dataset, record set `cr:RecordSet:main` contains proteins and attributes for downstream analysis.

In [ ]:
# Specify the record set(s) to extract (update for your dataset)
# For this dataset, the main record set has @id 'cr:RecordSet:main' (example, verify from above overview)
MAIN_RECORD_SET_ID = 'cr:RecordSet:main'  # replace with actual @id from the overview if needed

# If there are more record sets, list their @ids here as well
record_set_ids = [MAIN_RECORD_SET_ID]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Display the column names of the main record set
print(f"Columns in record set {MAIN_RECORD_SET_ID}:")
print(dataframes[MAIN_RECORD_SET_ID].columns.tolist())

dataframes[MAIN_RECORD_SET_ID].head()

## 4. Exploratory Data Analysis (EDA)

We'll perform some basic data processing on the main record set, such as filtering proteins by peptide counts, normalizing a numeric field, and grouping by a key categorical attribute. Make sure to refer to columns/fields using their `@id`.

In [ ]:
# Choose a numeric field to analyze (e.g., number of peptides or MW)
# List numeric columns to choose from
print("Available numeric columns:")
numeric_columns = dataframes[MAIN_RECORD_SET_ID].select_dtypes(include='number').columns.tolist()
print(numeric_columns)

NUMERIC_FIELD_ID = 'cr:column:peptide_count'  # Replace with correct numeric field @id, e.g., 'cr:column:MW' for molecular weight

# Set a threshold for filtering
threshold = 10
df = dataframes[MAIN_RECORD_SET_ID]
if NUMERIC_FIELD_ID in df.columns:
    filtered_df = df[df[NUMERIC_FIELD_ID] > threshold].copy()
    print(f"Number of records with {NUMERIC_FIELD_ID} > {threshold}: {len(filtered_df)}")
    print(filtered_df.head())
    
    # Normalize the numeric field
    col_norm = f"{NUMERIC_FIELD_ID}_normalized"
    filtered_df[col_norm] = (filtered_df[NUMERIC_FIELD_ID] - filtered_df[NUMERIC_FIELD_ID].mean()) / filtered_df[NUMERIC_FIELD_ID].std()
    print(f"\nNormalized {NUMERIC_FIELD_ID} for filtered records:")
    print(filtered_df[[NUMERIC_FIELD_ID, col_norm]].head())
    
    # Group by a categorical field (e.g., 'cr:column:accession') if available
    GROUP_FIELD_ID = 'cr:column:protein_family'  # Example, replace with available group field @id if appropriate
    if GROUP_FIELD_ID in filtered_df.columns:
        grouped_df = filtered_df.groupby(GROUP_FIELD_ID)[NUMERIC_FIELD_ID].mean().reset_index()
        print(f"\nGrouped mean {NUMERIC_FIELD_ID} by {GROUP_FIELD_ID}:")
        print(grouped_df.head())
else:
    print(f"Numeric field {NUMERIC_FIELD_ID} not found. Available columns: {df.columns.tolist()}")

## 5. Visualization

Visualize the distribution of the peptide count and the normalized values. If a group field was used, also show group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if NUMERIC_FIELD_ID in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[NUMERIC_FIELD_ID].dropna(), bins=30, kde=True)
    plt.xlabel(NUMERIC_FIELD_ID)
    plt.title(f'Distribution of {NUMERIC_FIELD_ID}')
    plt.show()

    if f"{NUMERIC_FIELD_ID}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{NUMERIC_FIELD_ID}_normalized"].dropna(), bins=30, kde=True, color='orange')
        plt.xlabel(f"{NUMERIC_FIELD_ID}_normalized")
        plt.title(f'Normalized {NUMERIC_FIELD_ID} (filtered)')
        plt.show()

    # If grouping by GROUP_FIELD_ID, plot as bar chart
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df.sort_values(NUMERIC_FIELD_ID, ascending=False).head(10), x=GROUP_FIELD_ID, y=NUMERIC_FIELD_ID)
        plt.xticks(rotation=45)
        plt.title(f'Mean {NUMERIC_FIELD_ID} by {GROUP_FIELD_ID} (top 10)')
        plt.show()


## 6. Conclusion

We've loaded the mass spectrometry dataset package in Croissant/FAIR² format using `mlcroissant`, reviewed its structure, and performed a basic exploratory analysis. For further work, consider protein-specific filters, advanced visualization, or integration of additional annotations by referencing specific `@id` fields for robust, reproducible data science workflows.